## Tested personality traits ✍️

In this guide we go through how to configure the personality traits that are annotated and evaluated with Feedback Forensics. We go through the traits *tested by default*, how to *set custom traits* and how to *automatically generate new traits*.

### Default personality traits tested

Below are the principles tested by default when using `ff-annotate`.

In [ ]:
from feedback_forensics.tools.ff_annotate import get_default_principles

get_default_principles()

### Annotating custom personality traits

To annotate the data with our own custom personality traits we use the `--principles` argument of `ff-annotate`. Note that the value will be loaded by `json.loads()`. Thus, principles should be surrounded by double quotes (`"`) and otherwise comply with json list format. To avoid testing the standard principles as well, simply set the `--principle-version` argument to an empty string `""`.

In [ ]:
!ff-annotate -d="../../data/input/example.csv" --principles '["Select the response that is more confident","Select the response that is more friendly"]' --principles-version ""

### Position debiasing

AI annotators can be biased towards the response shown first (or second). By default, `ff-annotate` shows the two responses of each comparison once, in a random (but reproducible) order. With `--position-debias`, each comparison is annotated twice, once in each order, and only votes that agree across both orders are kept. Votes that change with the order are stored without a preference (`no_pref_reason: "position_inconsistent"`), and the votes for each order are saved to `results/041_votes_per_comparison_per_order.csv`. This doubles the annotation cost.

```
ff-annotate -d data.csv --position-debias
```

### API usage and cost

Every API call is logged to `api_calls.jsonl` in the output directory (token counts incl. reasoning tokens, cost, finish reason, duration and errors); add `--log-api-calls` to also log the full prompts and responses. Totals are logged at the end of a run and saved to `results/080_api_usage_summary.json`, together with the number of failed, truncated and unparsable calls (which result in invalid votes). The cost is reported by the provider for OpenRouter models (`openrouter/...`), for which the cost at OpenRouter list prices is also computed. For reasoning models, use `--max-output-tokens` (default 3000, including reasoning tokens) and `--reasoning-effort` to avoid truncated responses.

### Automatically detecting personality traits

You can also use [Inverse Constitutional AI](https://github.com/rdnfn/icai) (ICAI) to automatically detect personality traits that may help explain your dataset. See the `s0_skip_principle_generation=false` to allow the pipeline to automatically generate principles that appear to separate responses in the dataset.

Principle discovery is the only part of the pipeline that uses ICAI, which is an optional dependency: install it with `pip install "feedback-forensics[discovery]"`. The discovered principles (in `results/030_distilled_principles_per_cluster.json` of the ICAI output) can then be passed to `ff-annotate` via the `--principles` argument.

In [ ]:
!icai-exp data_path="../../data/input/example.csv" s0_added_standard_principles_to_test="[v2]" annotator.skip=true s0_skip_principle_generation=false